In [1]:
import pandas as pd
import sqlalchemy as sa

In [2]:
server = "SDVWEDWSHSS01"
database = "SupplyChainAnalyticsDB"

connection_string = (
    "mssql+pyodbc://@{server}/{db}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
).format(server=server, db=database)

engine = sa.create_engine(connection_string)


In [3]:
sql_query = """
WITH OTIF_Base_Data AS (
    SELECT 
        [Sales order],
        [SO Line],
        [SO create date],
        [Mat_Avl_Date_OTIF],
        Material, 
        [Material description],
        [ABC Indicator],
        Plant,
        Ship_To,
        [Sold-to party], 
        [Net value_y] AS [Net_Value(Header Level od Document)], 
        [Net value_x] AS [Net_Value(Item Level at Document)], 
        [Document Currency] AS [Local Currency],
        [Net weight] AS Ordered_Quantity, 
        [Sales Organization], 
        Reporting_Date AS [Requested Delivery Date], 
        CSR,
        Customer_Pickup, 
        [OTIF_HIT/MISS],
        [OTIF_Type], 
        Overdeliv_Tolerance_OTIF,
        Underdel_Tolerance_OTIF, 
        [Confirmed Quantity_OTIF] AS First_Confirmed_Quantity, 
        Time_Factor, 
        [Delivery Date_y_OTIF] AS [Delivery_Date For Solenis], 
        Act_Gd_Mvmnt_Date_OTIF AS [Delivery_Date For Sigura and Diversey (Actual Goods Movement)],
        TASKI_Indicator AS [TASKI Machine Indicator], 
        [Delivery Number], 
        [Delivery Created On], 
        Agg_Qty AS Orderd_Qty_y
    FROM [SupplyChainAnalyticsDB].dbo.SCA_N_OTIF_v3 a
    WHERE Reporting_Date >= '2023-09-01'
      AND Reporting_Date <  '2026-02-01'
      AND [Customer group] <> '99'
      AND [Item category] <> 'ZFTG'
      AND [Order reason] NOT IN ('Z70', 'Y42')
      AND [Customer_Pickup] = 'No'
      AND [OTIF_Type] = 'Internal Plant OTIF'
      AND EXISTS (
          SELECT 1
          FROM SupplyChainAnalyticsDB.dbo.SCA_Plant_Master b
          WHERE a.Plant = b.Plant
            AND b.[Plant Region] = 'NAM'
      )
),

VBAP AS (
    SELECT DISTINCT
        REPLACE(LTRIM(REPLACE(a.[VBELN: (PK) Sales Document],'0',' ')),' ','0') AS [Sales Order],
        REPLACE(LTRIM(REPLACE(a.[POSNR: (PK) Sales Document Item],'0',' ')),' ','0') AS [SOItem],
        a.[MATNR: Material Number] AS Material,
        a.[NTGEW: Net Weight of the Item],
        a.[GEWEI: Unit of Weight],
        a.[PSTYV: Sales Document Item Category] AS ItemCategory,
        a.[WERKS: Plant (Own or External)] AS Plant,
        b.[Location Type for Metrics],
        a.[NETWR: Net value of the order item in document currency] AS NetValue_in_Local_Currency,
        a.[WAERK: SD Document Currency] AS SD_Document_Currency,
        CASE 
            WHEN a.[GEWEI: Unit of Weight] = 'KG' THEN a.[NTGEW: Net Weight of the Item]
            WHEN a.[GEWEI: Unit of Weight] = 'G'  THEN a.[NTGEW: Net Weight of the Item] * 0.001
            WHEN a.[GEWEI: Unit of Weight] = 'LB' THEN a.[NTGEW: Net Weight of the Item] * 0.453592
            WHEN a.[GEWEI: Unit of Weight] = 'OZ' THEN a.[NTGEW: Net Weight of the Item] * 0.0283495
            ELSE a.[NTGEW: Net Weight of the Item]
        END AS NtWtInKGs,
        a.[MEINS: Base Unit of Measure] AS BaseUOM,
        [KWMENG: Cumulative Order Quantity in Sales Units] *
        (CAST([UMVKZ: Numerator (factor) for conversion of sales quantity into SKU] AS FLOAT) /
         NULLIF([UMVKN: Denominator (Divisor) for Conversion of Sales Qty into SKU], 0)) AS Base_Quanity
    FROM [LIB_EDW_RTP].[bv].[VBAP: Sales Document: Item Data] a
    LEFT JOIN (
        SELECT DISTINCT Plant, [Location Type for Metrics], [Legacy Firm]
        FROM SupplyChainAnalyticsDB.dbo.SCA_Plant_Master
    ) b
        ON a.[WERKS: Plant (Own or External)] = b.Plant
    WHERE EXISTS (
        SELECT 1
        FROM OTIF_Base_Data c
        WHERE REPLACE(LTRIM(REPLACE(a.[VBELN: (PK) Sales Document],'0',' ')),' ','0') = c.[Sales order]
          AND REPLACE(LTRIM(REPLACE(a.[POSNR: (PK) Sales Document Item],'0',' ')),' ','0') = c.[SO Line]
    )
),

LIPS AS (
    SELECT
        REPLACE(LTRIM(REPLACE(LIPS.[VGBEL: Document Number of Reference Document],'0',' ')),' ','0') AS [Sales Order],
        REPLACE(LTRIM(REPLACE(LIPS.[VGPOS: Item number of the reference item],'0',' ')),' ','0') AS [SOItem],
        LIPS.[MATNR: Material Number] AS Material,
        LIPS.[WERKS: Plant] AS Plant,
        LIPS.[MEINS: Base Unit of Measure] AS Base_UOM,
        SUM(
            CASE 
                WHEN LIPS.[GEWEI: Unit of Weight] = 'KG' THEN LIPS.[NTGEW: Net Weight]
                WHEN LIPS.[GEWEI: Unit of Weight] = 'G'  THEN LIPS.[NTGEW: Net Weight] * 0.001
                WHEN LIPS.[GEWEI: Unit of Weight] = 'LB' THEN LIPS.[NTGEW: Net Weight] * 0.453592
                WHEN LIPS.[GEWEI: Unit of Weight] = 'OZ' THEN LIPS.[NTGEW: Net Weight] * 0.0283495
                ELSE LIPS.[NTGEW: Net Weight]
            END
        ) AS Delivered_NtWtInKGs,
        SUM(
            LIPS.[LFIMG: Actual quantity delivered (in sales units)] *
            (CAST(LIPS.[UMVKZ: Numerator (factor) for conversion of sales quantity into SKU] AS FLOAT) /
             NULLIF(LIPS.[UMVKN: Denominator (Divisor) for Conversion of Sales Qty into SKU], 0))
        ) AS Delivered_Quantity_in_Base_UOM
    FROM [LIB_EDW_RTP].[bv].[LIPS: SD document: Delivery: Item data] LIPS
    WHERE EXISTS (
        SELECT 1
        FROM OTIF_Base_Data c
        WHERE REPLACE(LTRIM(REPLACE(LIPS.[VGBEL: Document Number of Reference Document],'0',' ')),' ','0') = c.[Sales order]
          AND REPLACE(LTRIM(REPLACE(LIPS.[VGPOS: Item number of the reference item],'0',' ')),' ','0') = c.[SO Line]
    )
    GROUP BY
        LIPS.[VGBEL: Document Number of Reference Document],
        LIPS.[VGPOS: Item number of the reference item],
        LIPS.[MATNR: Material Number],
        LIPS.[WERKS: Plant],
        LIPS.[MEINS: Base Unit of Measure]
)

SELECT
    a.*,
    b.NtWtInKGs AS Ordered_Qty_in_Kgs,
    b.BaseUOM AS Ordered_Quantity_Base_UOM,
    b.Base_Quanity AS Ordered_in_Base_UOM,
    b.NetValue_in_Local_Currency AS Ordered_Value_in_Currency,
    b.SD_Document_Currency AS Local_Currency_Item,
    c.Delivered_NtWtInKGs AS Delivered_Qty_in_Kgs,
    c.Delivered_Quantity_in_Base_UOM,
    c.Base_UOM,
    d.[Customer Name],
    d.City,
    d.Country,
    d.[State - Province],
    g.[Division of Business Name],
    g.[Product Line Name] AS Material_Product_line,
    g.MATERIAL_TYPE,
    g.[Material Base Code Desc]
FROM OTIF_Base_Data a
LEFT JOIN VBAP b
    ON a.[Sales order] = b.[Sales Order]
   AND a.[SO Line]     = b.[SOItem]
LEFT JOIN LIPS c
    ON a.[Sales order] = c.[Sales Order]
   AND a.[SO Line]     = c.[SOItem]
LEFT JOIN (
    SELECT DISTINCT
        REPLACE(LTRIM(REPLACE(Customer,'0',' ')),' ','0') AS Customer_Key,
        [Customer Name],
        City,
        [State - Province],
        Country
    FROM LIB_EDW_RTP.dim.Customer
) d
    ON REPLACE(LTRIM(REPLACE(a.Ship_To,'0',' ')),' ','0') = d.Customer_Key
LEFT JOIN (
    SELECT DISTINCT
        Material,
        [Division of Business Name],
        [Product Line Name],
        MATERIAL_TYPE,
        [Material Base Code Desc]
    FROM SupplyChainAnalyticsDB.dbo.SCA_Material_Master
) g
    ON a.Material = g.Material;

"""


In [4]:
df_raw = pd.read_sql(sql_query, engine, chunksize=100000)

df_raw = pd.concat(df_raw, ignore_index=True)


In [5]:
df_raw.describe()

,Mat_Avl_Date_OTIF,Net_Value(Header Level od Document),Net_Value(Item Level at Document),Ordered_Quantity,Overdeliv_Tolerance_OTIF,Underdel_Tolerance_OTIF,First_Confirmed_Quantity,Delivery_Date For Solenis,Delivery_Date For Sigura and Diversey (Actual Goods Movement),Orderd_Qty_y,Ordered_Qty_in_Kgs,Ordered_in_Base_UOM,Ordered_Value_in_Currency,Delivered_Qty_in_Kgs,Delivered_Quantity_in_Base_UOM
count,1253105,1.360238e+06,1.360238e+06,1.360238e+06,1.360238e+06,1.360238e+06,1.360238e+06,1253105,1233911,1.253106e+06,1.360181e+06,1.360181e+06,1.360181e+06,1.278828e+06,1.278828e+06
mean,2025-03-03 08:18:23.064788,4.631881e+04,3.564234e+03,1.662006e+03,1.060755e+00,1.071830e+00,8.347765e+02,2025-03-08 18:55:15.968573,2025-03-06 08:25:19.484630,1.388528e+03,1.258508e+03,1.054237e+03,3.563226e+03,9.909880e+02,7.944086e+02
min,2022-12-15 00:00:00,0.000000e+00,-7.472000e+01,1.000000e-03,0.000000e+00,0.000000e+00,0.000000e+00,2023-02-03 00:00:00,2022-12-27 00:00:00,0.000000e+00,0.000000e+00,0.000000e+00,-7.472000e+01,0.000000e+00,1.666667e-01
25%,2024-09-19 00:00:00,1.166950e+03,1.137000e+02,3.000000e+01,0.000000e+00,0.000000e+00,1.000000e+00,2024-09-25 00:00:00,2024-09-21 00:00:00,3.000000e+01,1.439882e+01,2.000000e+00,1.137300e+02,1.398016e+01,2.000000e+00
50%,2025-03-17 00:00:00,1.544510e+04,4.327200e+02,1.187500e+02,0.000000e+00,0.000000e+00,3.000000e+00,2025-03-21 00:00:00,2025-03-20 00:00:00,1.146690e+02,5.502978e+01,9.000000e+00,4.328100e+02,5.335149e+01,9.000000e+00
75%,2025-08-04 00:00:00,7.652774e+04,1.993360e+03,6.319500e+02,0.000000e+00,0.000000e+00,1.500000e+01,2025-08-08 00:00:00,2025-08-06 00:00:00,6.250000e+02,3.051223e+02,4.400000e+01,1.994150e+03,2.951931e+02,4.400000e+01
max,2029-04-26 00:00:00,6.434240e+06,2.117813e+08,4.898700e+07,9.900000e+01,9.900000e+01,1.000000e+07,2202-03-20 00:00:00,2026-02-04 00:00:00,2.250938e+07,4.898700e+07,4.898700e+07,2.117813e+08,2.250938e+07,2.968000e+05
std,NaN,6.296328e+04,1.915097e+05,1.064483e+05,3.282773e+00,3.330512e+00,1.079126e+04,NaN,NaN,3.876509e+04,1.019620e+05,9.733830e+04,1.915119e+05,2.971810e+04,3.686913e+03


In [6]:
df_raw.dtypes

Sales order                                                                 str
SO Line                                                                     str
SO create date                                                           object
Mat_Avl_Date_OTIF                                                datetime64[us]
Material                                                                    str
Material description                                                        str
ABC Indicator                                                               str
Plant                                                                       str
Ship_To                                                                     str
Sold-to party                                                               str
Net_Value(Header Level od Document)                                     float64
Net_Value(Item Level at Document)                                       float64
Local Currency                          

In [7]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 1360238 entries, 0 to 1360237
Data columns (total 46 columns):
 #   Column                                                         Non-Null Count    Dtype         
---  ------                                                         --------------    -----         
 0   Sales order                                                    1360238 non-null  str           
 1   SO Line                                                        1360238 non-null  str           
 2   SO create date                                                 1360238 non-null  object        
 3   Mat_Avl_Date_OTIF                                              1253105 non-null  datetime64[us]
 4   Material                                                       1360238 non-null  str           
 5   Material description                                           1360238 non-null  str           
 6   ABC Indicator                                                  1358562 non-null  str 

In [8]:
df_raw["Ordered_Quantity_Base_UOM"].head(10)

0    KG
1    SU
2    SU
3    SU
4    SU
5    KG
6    KG
7    SU
8    SU
9    SU
Name: Ordered_Quantity_Base_UOM, dtype: str

In [10]:
import numpy as np
import pandas as pd
import lightgbm as lgb
 
from sklearn.metrics import (
    roc_auc_score, accuracy_score,
    precision_score, recall_score, f1_score,
    confusion_matrix
)
 
RANDOM_STATE = 42
 
TARGET_RAW = "OTIF_HIT/MISS"
TARGET_COL = "otif_numeric"
SPLIT_DATE_COL = "Requested Delivery Date"   # you confirmed this
 
DATE_COLS = ["SO create date", "Mat_Avl_Date_OTIF", "Requested Delivery Date"]
 
LEAKAGE_COLS = [
    "Delivery_Date For Solenis",
    "Delivery_Date For Sigura and Diversey (Actual Goods Movement)",
    "Delivery Number",
    "Delivery Created On",
    "Delivered_Qty_in_Kgs",
    "Delivered_Quantity_in_Base_UOM",
    "OTIF_Type",
    "Time_Factor",
]
 
WANTED_28 = [
    "f_request_lead_days","f_material_lead_days","f_lead_gap_days","f_tight_ratio",
    "f_is_tight_order","f_is_extremely_tight","f_so_to_rdd_days","f_mat_avail_to_rdd_days",
    "f_mat_ready_after_rdd","f_unit_price_log","f_mat_total_orders_log","f_critical_negative_gap",
    "f_mild_negative_gap","f_large_positive_gap","f_tight_x_pressure","f_high_plant_risk",
    "f_risk_stack","f_otif_risk_score","f_gap_bin","f_plant_miss_rate","f_customer_miss_rate",
    "f_material_miss_rate","f_gap_x_pressure","f_bu_miss_rate","f_so_woy_sin","f_so_woy_cos",
    "f_rdd_woy_sin","f_rdd_woy_cos"
]
WANTED_28 = list(dict.fromkeys(WANTED_28))

In [11]:
# ===============================
# STEP 1: CHECK MISSING VALUES
# ===============================

print("Total rows:", len(df_raw))
print("\n")

# Overall missing summary
missing_summary = (
    df_raw.isna()
      .sum()
      .to_frame("missing_count")
      .assign(missing_pct=lambda x: (x["missing_count"] / len(df_raw)) * 100)
      .sort_values("missing_pct", ascending=False)
)

print("=== Missing Summary (All Columns) ===")
print(missing_summary[missing_summary["missing_count"] > 0])
print("\n")

# Specifically check DATE columns
print("=== Date Column Missing ===")
for col in DATE_COLS:
    if col in df_raw.columns:
        print(f"{col}: {df_raw[col].isna().sum()} missing ({df_raw[col].isna().mean()*100:.2f}%)")
    else:
        print(f"{col}: NOT PRESENT in dataframe")


Total rows: 1360238


=== Missing Summary (All Columns) ===
                                                    missing_count  missing_pct
Delivery_Date For Sigura and Diversey (Actual G...         126327     9.287125
Mat_Avl_Date_OTIF                                          107133     7.876048
Delivery_Date For Solenis                                  107133     7.876048
Orderd_Qty_y                                               107132     7.875975
Delivery Number                                             94628     6.956724
Delivery Created On                                         94628     6.956724
Delivered_Quantity_in_Base_UOM                              81410     5.984982
Base_UOM                                                    81410     5.984982
Delivered_Qty_in_Kgs                                        81410     5.984982
Material_Product_line                                       46900     3.447926
Division of Business Name                                   46900     3

In [12]:
# =====================================
# CLEANING BLOCK (Corrected + Safe)
# =====================================

# -----------------------------
# 1️⃣ Drop leakage columns
# -----------------------------
df_raw.drop(columns=[c for c in LEAKAGE_COLS if c in df_raw.columns],
            inplace=True,
            errors="ignore")

# -----------------------------
# 2️⃣ Drop merge artifact
# -----------------------------
if "Orderd_Qty_y" in df_raw.columns:
    df_raw.drop(columns=["Orderd_Qty_y"], inplace=True)

# -----------------------------
# 3️⃣ Fill categorical missing with "Unknown"
# -----------------------------
categorical_cols = [
    "Division of Business Name",
    "MATERIAL_TYPE",
    "Material_Product_line",
    "ABC Indicator",
    "Base_UOM",
    "Material Base Code Desc",
    "Ordered_Quantity_Base_UOM",  # ✅ THIS IS CATEGORICAL (UNIT)
    "Local_Currency_Item"
]

for col in categorical_cols:
    if col in df_raw.columns:
        df_raw[col] = df_raw[col].fillna("Unknown")

# -----------------------------
# 4️⃣ Fill small numeric missing with MEDIAN
# -----------------------------
numeric_cols = [
    "Ordered_Value_in_Currency",
    "Ordered_Qty_in_Kgs",
    "Ordered_in_Base_UOM",
]

for col in numeric_cols:
    if col in df_raw.columns:
        df_raw[col] = df_raw[col].fillna(df_raw[col].median())

# -----------------------------
# 5️⃣ Create Mat_Avl missing flag (DO NOT IMPUTE DATE)
# -----------------------------
if "Mat_Avl_Date_OTIF" in df_raw.columns:
    df_raw["f_mat_avl_missing"] = df_raw["Mat_Avl_Date_OTIF"].isna().astype("Int64")

print("Cleaning complete.")
print("\nRemaining missing % (top 10):")
print(
    df_raw.isna()
      .mean()
      .sort_values(ascending=False)
      .head(10)
)

Cleaning complete.

Remaining missing % (top 10):
Mat_Avl_Date_OTIF       0.07876
Sales order             0.00000
SO Line                 0.00000
SO create date          0.00000
Material                0.00000
Material description    0.00000
ABC Indicator           0.00000
Plant                   0.00000
Ship_To                 0.00000
Sold-to party           0.00000
dtype: float64


In [13]:
def _parse_dates(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in DATE_COLS:
        if c in out.columns:
            out[c] = pd.to_datetime(out[c], errors="coerce", dayfirst=True)
    return out
 
def _drop_leakage(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop(columns=[c for c in LEAKAGE_COLS if c in df.columns], errors="ignore")
 
def _make_target(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out[TARGET_COL] = (
        out[TARGET_RAW].astype(str).str.strip().str.lower().map({"hit": 1, "miss": 0})
    )
    out = out.dropna(subset=[TARGET_COL])
    out[TARGET_COL] = out[TARGET_COL].astype(int)
    return out
 
def add_safe_features(df_in: pd.DataFrame) -> pd.DataFrame:
    out = df_in.copy()
 
    num_cols = [
        "Ordered_Qty_in_Kgs","Ordered_Quantity",
        "Ordered_Value_in_Currency","Net_Value(Item Level at Document)"
    ]
    for c in num_cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
 
    qty_col = "Ordered_Qty_in_Kgs" if "Ordered_Qty_in_Kgs" in out.columns else "Ordered_Quantity"
    val_col = "Ordered_Value_in_Currency" if "Ordered_Value_in_Currency" in out.columns else "Net_Value(Item Level at Document)"
 
    out["f_so_to_rdd_days"] = (out["Requested Delivery Date"] - out["SO create date"]).dt.days
    out["f_so_to_mat_avail_days_from_dates"] = (out["Mat_Avl_Date_OTIF"] - out["SO create date"]).dt.days
    out["f_mat_avail_to_rdd_days"] = (out["Requested Delivery Date"] - out["Mat_Avl_Date_OTIF"]).dt.days
    out["f_mat_ready_after_rdd"] = (out["f_mat_avail_to_rdd_days"] < 0).astype("Int64")
 
    out["f_request_lead_days"] = out["f_so_to_rdd_days"]
    out["f_material_lead_days"] = out["f_so_to_mat_avail_days_from_dates"]
 
    out["f_lead_gap_days"] = out["f_request_lead_days"] - out["f_material_lead_days"]
    out["f_tight_ratio"] = out["f_request_lead_days"] / (out["f_material_lead_days"] + 1.0)
    out["f_is_tight_order"] = (out["f_tight_ratio"] < 1.0).astype("Int64")
    out["f_is_extremely_tight"] = (out["f_tight_ratio"] < 0.75).astype("Int64")
 
    out["f_critical_negative_gap"] = (out["f_lead_gap_days"] < -3).astype("Int64")
    out["f_mild_negative_gap"] = ((out["f_lead_gap_days"] < 0) & (out["f_lead_gap_days"] >= -3)).astype("Int64")
    out["f_large_positive_gap"] = (out["f_lead_gap_days"] > 7).astype("Int64")
 
    out["_qty"] = pd.to_numeric(out.get(qty_col), errors="coerce")
    out["_val"] = pd.to_numeric(out.get(val_col), errors="coerce")
    out["f_unit_price_log"] = np.log1p((out["_val"] / (out["_qty"] + 1e-9)).clip(lower=0))
 
    # seasonality on SO
    so = out["SO create date"]
    w = so.dt.isocalendar().week.astype("Int64").fillna(0).astype(float)
    out["f_so_woy_sin"] = np.sin(2 * np.pi * (w / 52.0))
    out["f_so_woy_cos"] = np.cos(2 * np.pi * (w / 52.0))
 
    # seasonality on RDD
    rdd = out["Requested Delivery Date"]
    rw = rdd.dt.isocalendar().week.astype("Int64").fillna(0).astype(float)
    out["f_rdd_woy_sin"] = np.sin(2 * np.pi * (rw / 52.0))
    out["f_rdd_woy_cos"] = np.cos(2 * np.pi * (rw / 52.0))
 
    out = out.drop(columns=[c for c in ["_qty", "_val"] if c in out.columns])
    return out
 
def build_miss_rate_maps_recent(train_df: pd.DataFrame,
                                alpha: float = 20.0,
                                recent_months: int = 6):
 
    cutoff = train_df[SPLIT_DATE_COL].max() - pd.DateOffset(months=recent_months)
    recent_df = train_df[train_df[SPLIT_DATE_COL] >= cutoff].copy()
 
    global_miss = 1.0 - recent_df[TARGET_COL].mean()
 
    group_cols = [
        ("Ship_To", "f_customer_miss_rate"),
        ("Material", "f_material_miss_rate"),
        ("Plant", "f_plant_miss_rate"),
        ("Division of Business Name", "f_bu_miss_rate"),
    ]
 
    maps = {}
    for gcol, feat in group_cols:
        if gcol not in recent_df.columns:
            continue
 
        stats = recent_df.groupby(gcol)[TARGET_COL].agg(["count", "mean"])
        stats["miss_rate"] = 1.0 - stats["mean"]
        stats["miss_rate_smooth"] = (
            stats["miss_rate"] * stats["count"] + alpha * global_miss
        ) / (stats["count"] + alpha)
 
        maps[feat] = stats["miss_rate_smooth"].to_dict()
 
    return global_miss, maps
 
def apply_miss_rate_maps(df: pd.DataFrame, global_miss: float, maps: dict) -> pd.DataFrame:
    out = df.copy()
    col_map = {
        "f_customer_miss_rate": "Ship_To",
        "f_material_miss_rate": "Material",
        "f_plant_miss_rate": "Plant",
        "f_bu_miss_rate": "Division of Business Name",
    }
    for feat, gcol in col_map.items():
        if gcol in out.columns and feat in maps:
            out[feat] = out[gcol].map(maps[feat]).fillna(global_miss)
        else:
            out[feat] = global_miss
    return out
 
def apply_material_counts(train_df: pd.DataFrame, test_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    tr = train_df.copy()
    te = test_df.copy()
    if "Material" in tr.columns:
        mat_counts = tr.groupby("Material").size()
        tr["f_mat_total_orders_log"] = np.log1p(tr["Material"].map(mat_counts).fillna(0))
        te["f_mat_total_orders_log"] = np.log1p(te["Material"].map(mat_counts).fillna(0))
    else:
        tr["f_mat_total_orders_log"] = 0.0
        te["f_mat_total_orders_log"] = 0.0
    return tr, te
 
def apply_pressure_and_bins(df: pd.DataFrame, gap_q: float) -> pd.DataFrame:
    out = df.copy()
    pressure_cols = [c for c in ["f_plant_miss_rate", "f_bu_miss_rate"] if c in out.columns]
    if pressure_cols:
        out["f_pressure_proxy"] = out[pressure_cols].mean(axis=1).clip(0, 1)
        out["f_gap_x_pressure"] = out["f_lead_gap_days"] * (1.0 + out["f_pressure_proxy"])
        out["f_tight_x_pressure"] = out["f_is_extremely_tight"] * (1.0 + out["f_pressure_proxy"])
    else:
        out["f_gap_x_pressure"] = out["f_lead_gap_days"]
        out["f_tight_x_pressure"] = out["f_is_extremely_tight"]
 
    out["f_high_plant_risk"] = (out.get("f_plant_miss_rate", 0) > 0.25).astype("Int64")
    out["f_risk_stack"] = (out["f_high_plant_risk"] * out["f_is_extremely_tight"]).astype("Int64")
 
    out["f_otif_risk_score"] = (
        out["f_is_extremely_tight"].fillna(0)
        + out["f_critical_negative_gap"].fillna(0)
        + out["f_high_plant_risk"].fillna(0)
    ).astype(float)
 
    out["f_gap_bin"] = (out["f_lead_gap_days"] > gap_q).astype("Int64")
    return out

In [14]:
def month_range(start: str, end: str):
    """
    Yield monthly Periods from start to end inclusive.
    Input format: 'YYYY-MM'
    """
    cur = pd.Period(start, freq="M")
    endp = pd.Period(end, freq="M")
    while cur <= endp:
        yield cur
        cur += 1

In [15]:
# =========================
# PHASE 1: Pairwise priors (Recall boost)
# =========================
PAIRWISE_FEATS = [
    "f_mat_shipto_miss_rate",
    "f_plant_material_miss_rate",
    "f_plant_shipto_miss_rate",  # optional but safe
]
 
FEATURE_COLS = list(dict.fromkeys(WANTED_28 + PAIRWISE_FEATS))

In [16]:
def build_miss_rate_maps_recent(
    train_df: pd.DataFrame,
    alpha: float = 20.0,
    recent_months: int = 6,
    min_pair_count: int = 20,   # safety: avoid super-sparse pairs dominating
):
    """
    Builds smoothed miss-rate maps from RECENT slice of TRAIN ONLY.
 
    Returns:
      global_miss: float
      maps: dict[str, dict] feature_name -> mapping key -> smoothed_miss
            For pairwise maps, key is a tuple like (Material, Ship_To)
    """
    cutoff = train_df[SPLIT_DATE_COL].max() - pd.DateOffset(months=recent_months)
    recent_df = train_df[train_df[SPLIT_DATE_COL] >= cutoff].copy()
 
    # global miss from recent
    global_miss = 1.0 - recent_df[TARGET_COL].mean()
 
    maps = {}
 
    def _single_map(df, gcol: str):
        stats = df.groupby(gcol)[TARGET_COL].agg(["count", "mean"])
        stats["miss_rate"] = 1.0 - stats["mean"]
        stats["miss_rate_smooth"] = (
            stats["miss_rate"] * stats["count"] + alpha * global_miss
        ) / (stats["count"] + alpha)
        return stats["miss_rate_smooth"].to_dict()
 
    def _pair_map(df, c1: str, c2: str):
        # build tuple key
        keys = list(zip(df[c1].astype(str), df[c2].astype(str)))
        tmp = df[[TARGET_COL]].copy()
        tmp["_k"] = keys
 
        stats = tmp.groupby("_k")[TARGET_COL].agg(["count", "mean"])
        stats = stats[stats["count"] >= min_pair_count]  # drop tiny pairs
        if len(stats) == 0:
            return {}
 
        stats["miss_rate"] = 1.0 - stats["mean"]
        stats["miss_rate_smooth"] = (
            stats["miss_rate"] * stats["count"] + alpha * global_miss
        ) / (stats["count"] + alpha)
        return stats["miss_rate_smooth"].to_dict()
 
    # ---- single priors (you already had these)
    single_specs = [
        ("Ship_To", "f_customer_miss_rate"),
        ("Material", "f_material_miss_rate"),
        ("Plant", "f_plant_miss_rate"),
        ("Division of Business Name", "f_bu_miss_rate"),
    ]
    for gcol, feat in single_specs:
        if gcol in recent_df.columns:
            maps[feat] = _single_map(recent_df, gcol)
 
    # ---- pairwise priors (PHASE 1)
    # Material × Ship_To
    if ("Material" in recent_df.columns) and ("Ship_To" in recent_df.columns):
        maps["f_mat_shipto_miss_rate"] = _pair_map(recent_df, "Material", "Ship_To")
 
    # Plant × Material
    if ("Plant" in recent_df.columns) and ("Material" in recent_df.columns):
        maps["f_plant_material_miss_rate"] = _pair_map(recent_df, "Plant", "Material")
 
    # Plant × Ship_To (optional)
    if ("Plant" in recent_df.columns) and ("Ship_To" in recent_df.columns):
        maps["f_plant_shipto_miss_rate"] = _pair_map(recent_df, "Plant", "Ship_To")
 
    return float(global_miss), maps

# ----------------------------
# 3) REPLACE apply_miss_rate_maps (adds fallback hierarchy)
# ----------------------------
def apply_miss_rate_maps(df: pd.DataFrame, global_miss: float, maps: dict) -> pd.DataFrame:
    """
    Applies miss-rate priors with SAFE fallback hierarchy:
      pairwise -> single -> global
    """
    out = df.copy()
 
    # ---- single priors
    def _apply_single(feat: str, gcol: str):
        if gcol in out.columns and feat in maps and isinstance(maps[feat], dict) and len(maps[feat]) > 0:
            out[feat] = out[gcol].astype(str).map(maps[feat]).fillna(global_miss)
        else:
            out[feat] = global_miss
 
    _apply_single("f_customer_miss_rate", "Ship_To")
    _apply_single("f_material_miss_rate", "Material")
    _apply_single("f_plant_miss_rate", "Plant")
    _apply_single("f_bu_miss_rate", "Division of Business Name")
 
    # ---- pairwise priors with fallback to singles -> global
    def _apply_pair(feat: str, c1: str, c2: str, fallback_feat_a: str, fallback_feat_b: str):
        if (c1 in out.columns) and (c2 in out.columns) and (feat in maps) and isinstance(maps[feat], dict) and len(maps[feat]) > 0:
            keys = list(zip(out[c1].astype(str), out[c2].astype(str)))
            s = pd.Series(keys, index=out.index).map(maps[feat])
            # fallback: prefer the more specific single, then other, then global
            fb1 = out.get(fallback_feat_a, global_miss)
            fb2 = out.get(fallback_feat_b, global_miss)
            out[feat] = s.fillna(fb1).fillna(fb2).fillna(global_miss)
        else:
            # if pair map not available, fallback immediately
            out[feat] = out.get(fallback_feat_a, global_miss).fillna(out.get(fallback_feat_b, global_miss)).fillna(global_miss)
 
    # Material × Ship_To -> fallback to material + customer
    _apply_pair(
        feat="f_mat_shipto_miss_rate",
        c1="Material", c2="Ship_To",
        fallback_feat_a="f_material_miss_rate",
        fallback_feat_b="f_customer_miss_rate",
    )
 
    # Plant × Material -> fallback to plant + material
    _apply_pair(
        feat="f_plant_material_miss_rate",
        c1="Plant", c2="Material",
        fallback_feat_a="f_plant_miss_rate",
        fallback_feat_b="f_material_miss_rate",
    )
 
    # Plant × Ship_To -> fallback to plant + customer
    _apply_pair(
        feat="f_plant_shipto_miss_rate",
        c1="Plant", c2="Ship_To",
        fallback_feat_a="f_plant_miss_rate",
        fallback_feat_b="f_customer_miss_rate",
    )
 
    return out

In [17]:
CONGESTION_FEATS = [
    "f_plant_orders_7d",
    "f_plant_orders_30d",
    "f_material_orders_7d",
    "f_material_orders_30d",
    "f_shipto_orders_7d",
    "f_shipto_orders_30d",
]
 
def _build_rolling_count_table_train_only(
    train_df: pd.DataFrame,
    key_col: str,
    date_col: str,
    window_days: int,
    feat_name: str,
) -> pd.DataFrame:
    """
    Train-only rolling counts per key using daily aggregation + time-based rolling.
    LEAK-SAFE: excludes the current day by shifting 1 day.
    Returns: [key_col, date_col, feat_name]
    """
    td = train_df[[key_col, date_col]].dropna().copy()
    if td.empty:
        return pd.DataFrame(columns=[key_col, date_col, feat_name])
 
    td[date_col] = pd.to_datetime(td[date_col], errors="coerce").dt.floor("D")
    td = td.dropna(subset=[date_col])
    if td.empty:
        return pd.DataFrame(columns=[key_col, date_col, feat_name])
 
    # daily counts per key
    daily = (
        td.groupby([key_col, date_col])
          .size()
          .reset_index(name="daily_cnt")
          .sort_values([key_col, date_col])
    )
 
    # time-based rolling sum per key, then shift(1) to exclude current day (past-only)
    rolled = (
        daily.set_index(date_col)
             .groupby(key_col)["daily_cnt"]
             .rolling(f"{window_days}D", min_periods=1)
             .sum()
             .shift(1)
             .reset_index(name=feat_name)
    )
 
    rolled[feat_name] = rolled[feat_name].fillna(0.0).astype(float)
    rolled = rolled.sort_values([key_col, date_col])
    return rolled[[key_col, date_col, feat_name]]
 
 
def _apply_rolling_count_feature(
    df_any: pd.DataFrame,
    roll_table: pd.DataFrame,
    key_col: str,
    date_col: str,
    feat_name: str,
) -> pd.DataFrame:
    """
    Apply a precomputed train-only rolling table to any df (train or test)
    using merge_asof (past-only).
    Missing keys/dates => 0.
    """
    out = df_any.copy()
    out[date_col] = pd.to_datetime(out[date_col], errors="coerce").dt.floor("D")
 
    if roll_table.empty or out.empty:
        out[feat_name] = 0.0
        return out
 
    # merge_asof needs sorting
    out = out.sort_values([date_col, key_col])
    roll_table = roll_table.sort_values([date_col, key_col])
 
    merged = pd.merge_asof(
        out,
        roll_table,
        on=date_col,
        by=key_col,
        direction="backward",
        allow_exact_matches=True,  # ok because roll_table already excludes current day via shift(1)
    )
 
    merged[feat_name] = merged[feat_name].fillna(0.0).astype(float)
    return merged
 
 
def build_and_apply_congestion_features(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    date_col: str = "SO create date",
    windows=(7, 30),
    plant_col: str = "Plant",
    material_col: str = "Material",
    shipto_col: str = "Ship_To",
):
    """
    Build train-only congestion rolling tables and apply to train/test.
    Returns (train_out, test_out)
    """
    tr = train_df.copy()
    te = test_df.copy()
 
    # ensure date is datetime
    tr[date_col] = pd.to_datetime(tr[date_col], errors="coerce")
    te[date_col] = pd.to_datetime(te[date_col], errors="coerce")
 
    # PLANT
    if plant_col in tr.columns:
        for w in windows:
            feat = f"f_plant_orders_{w}d"
            rt = _build_rolling_count_table_train_only(tr, plant_col, date_col, w, feat)
            tr = _apply_rolling_count_feature(tr, rt, plant_col, date_col, feat)
            te = _apply_rolling_count_feature(te, rt, plant_col, date_col, feat)
    else:
        for w in windows:
            feat = f"f_plant_orders_{w}d"
            tr[feat] = 0.0
            te[feat] = 0.0
 
    # MATERIAL
    if material_col in tr.columns:
        for w in windows:
            feat = f"f_material_orders_{w}d"
            rt = _build_rolling_count_table_train_only(tr, material_col, date_col, w, feat)
            tr = _apply_rolling_count_feature(tr, rt, material_col, date_col, feat)
            te = _apply_rolling_count_feature(te, rt, material_col, date_col, feat)
    else:
        for w in windows:
            feat = f"f_material_orders_{w}d"
            tr[feat] = 0.0
            te[feat] = 0.0
 
    # SHIP_TO
    if shipto_col in tr.columns:
        for w in windows:
            feat = f"f_shipto_orders_{w}d"
            rt = _build_rolling_count_table_train_only(tr, shipto_col, date_col, w, feat)
            tr = _apply_rolling_count_feature(tr, rt, shipto_col, date_col, feat)
            te = _apply_rolling_count_feature(te, rt, shipto_col, date_col, feat)
    else:
        for w in windows:
            feat = f"f_shipto_orders_{w}d"
            tr[feat] = 0.0
            te[feat] = 0.0
 
    return tr, te

In [18]:
FEATURE_COLS = list(dict.fromkeys(WANTED_28 + PAIRWISE_FEATS + CONGESTION_FEATS))

In [19]:
# ---- Configure your source column names here ----
OVER_TOL_COL = "Overdeliv_Tolerance_OTIF"
UNDER_TOL_COL = "Underdel_Tolerance_OTIF"
 
def add_tolerance_risk_features(
    df: pd.DataFrame,
    *,
    gap_col: str = "f_lead_gap_days",
    extremely_tight_col: str = "f_is_extremely_tight",
    strict_band_threshold: float = 0.05,   # 5% as you suggested
    clip_band_min: float = 0.0,
    clip_band_max: float = 1.0,
) -> pd.DataFrame:
    """
    Adds tolerance-based risk features:
      - f_tolerance_band
      - f_strict_tolerance
      - f_strict_x_tight
      - f_tolerance_x_gap
 
    Leak-safe: uses only row-level tolerance fields + precomputed gap/tight features.
    No target, no future-looking aggregations.
    """
    out = df.copy()
 
    # 1) Ensure numeric tolerance inputs (handle strings, blanks)
    over = pd.to_numeric(out.get(OVER_TOL_COL), errors="coerce")
    under = pd.to_numeric(out.get(UNDER_TOL_COL), errors="coerce")
 
    # If tolerances are stored as whole percentages (e.g., 5 meaning 5%), normalize to 0.05
    # Heuristic: if typical values > 1, assume percent points.
    if np.nanpercentile(over.dropna(), 90) > 1 or np.nanpercentile(under.dropna(), 90) > 1:
        over = over / 100.0
        under = under / 100.0
 
    # 2) f_tolerance_band = over + under (clipped)
    band = (over.fillna(0.0) + under.fillna(0.0)).astype(float)
    band = band.clip(lower=clip_band_min, upper=clip_band_max)
    out["f_tolerance_band"] = band
 
    # 3) f_strict_tolerance = 1 if band < threshold else 0
    # If tolerance is missing (both 0 via fillna), it becomes strict (band=0) -> 1.
    out["f_strict_tolerance"] = (out["f_tolerance_band"] < strict_band_threshold).astype(np.int8)
 
    # 4) f_strict_x_tight = strict * extremely_tight
    # Ensure extremely tight is 0/1 int
    tight = pd.to_numeric(out.get(extremely_tight_col), errors="coerce").fillna(0).astype(np.int8)
    out[extremely_tight_col] = tight
    out["f_strict_x_tight"] = (out["f_strict_tolerance"] * out[extremely_tight_col]).astype(np.int8)
 
    # 5) f_tolerance_x_gap
    # Interpretation: strict tolerance amplifies risk when gap is negative (late risk),
    # flexible tolerance dampens it. Use |negative gap| * strictness proxy.
    # This is continuous + monotonic-friendly for GBDT.
    gap = pd.to_numeric(out.get(gap_col), errors="coerce").fillna(0.0).astype(float)
    out[gap_col] = gap
 
    negative_gap_magnitude = (-gap).clip(lower=0.0)  # if gap=-3 => 3, if gap=5 => 0
    # strictness proxy: smaller band => bigger multiplier. Add epsilon to avoid div by 0.
    strictness_proxy = 1.0 / (out["f_tolerance_band"] + 1e-6)
    # optional: cap extreme blow-ups if band can be 0
    strictness_proxy = strictness_proxy.clip(upper=1e6)
 
    out["f_tolerance_x_gap"] = (negative_gap_magnitude * strictness_proxy).astype(float)
 
    return out

In [20]:
NEW_TOL_FEATURES = [
    "f_tolerance_band",
    "f_strict_tolerance",
    "f_strict_x_tight",
    "f_tolerance_x_gap",
]
 
FEATURE_COLS = [c for c in FEATURE_COLS if c not in NEW_TOL_FEATURES] + NEW_TOL_FEATURES

In [21]:
QTY_COL = "Ordered_Qty_in_Kgs"      # change if your column differs
VALUE_COL = "f_unit_price_log"      # you said you already have price; if you have raw value, adjust below
 
def fit_qty_thresholds(train_df: pd.DataFrame, *, qty_col: str = QTY_COL) -> dict:
    """
    Fit-only-on-train thresholds to avoid leakage.
    Returns a dict you store in your bundle and reuse for OOT/inference.
    """
    qty = pd.to_numeric(train_df.get(qty_col), errors="coerce")
    # Use non-null, non-negative values for threshold
    qty = qty[(qty.notna()) & (qty >= 0)]
    q90 = float(np.nanpercentile(qty.values, 90)) if len(qty) else 0.0
    return {"qty_p90": q90}
 
def add_order_complexity_features(
    df: pd.DataFrame,
    *,
    thresholds: dict,
    qty_col: str = QTY_COL,
    extremely_tight_col: str = "f_is_extremely_tight",
    value_col: str = VALUE_COL,
) -> pd.DataFrame:
    """
    Adds:
      - f_qty_log
      - f_high_qty_flag (qty > train p90)
      - f_high_value_flag (value > train p90 of value_col)  [optional: if you don't want to fit, see note]
      - f_high_value_x_tight
 
    Leak-safe: qty/value are row-level order attributes.
    Thresholds must be learned on TRAIN ONLY.
    """
    out = df.copy()
 
    # ---- Quantity magnitude ----
    qty = pd.to_numeric(out.get(qty_col), errors="coerce").fillna(0.0)
    qty = qty.clip(lower=0.0)
    out["f_qty_log"] = np.log1p(qty).astype(float)
 
    # ---- High quantity flag using train-fitted 90th percentile ----
    qty_p90 = float(thresholds.get("qty_p90", 0.0))
    out["f_high_qty_flag"] = (qty > qty_p90).astype(np.int8)
 
    # ---- Tight flag (ensure 0/1) ----
    tight = pd.to_numeric(out.get(extremely_tight_col), errors="coerce").fillna(0).astype(np.int8)
    out[extremely_tight_col] = tight
 
    # ---- High value flag (train-fitted p90) ----
    # If your "value" is not unit_price_log but e.g. order_value = qty*price, compute that first.
    # Here: we use an existing numeric value feature column (like unit_price_log).
    val = pd.to_numeric(out.get(value_col), errors="coerce").fillna(0.0).astype(float)
 
    # If you also want to fit a value threshold (recommended), expect it in thresholds.
    val_p90 = thresholds.get("value_p90", None)
    if val_p90 is None:
        # fallback: no threshold provided => high value flag becomes 0/1 based on df median (no training fit)
        # Better: fit value_p90 on train (see below in section 2).
        val_p90 = float(np.nanmedian(val.values))
 
    out["f_high_value_flag"] = (val > float(val_p90)).astype(np.int8)
 
    # ---- Interaction: High value × Tight ----
    out["f_high_value_x_tight"] = (out["f_high_value_flag"] * out[extremely_tight_col]).astype(np.int8)
 
    return out

In [22]:
def _col_as_series(df: pd.DataFrame, col: str, default=0.0) -> pd.Series:
    """Always return a Series aligned to df.index."""
    if col in df.columns:
        s = df[col]
        if isinstance(s, pd.Series):
            return s
    return pd.Series(default, index=df.index)
 
def build_recent_smoothed_rate_map(
    train_df: pd.DataFrame,
    key_col: str,
    target_col: str,
    split_month_col: str = "split_month",
    recent_months: int = 6,
    alpha: float = 20.0,         # smoothing strength
    global_fallback: float = None
):
    """
    Train-only artifact: recent miss rate map with Bayesian smoothing.
    miss rate = P(target==0)
    """
    d = train_df.copy()
    if split_month_col not in d.columns:
        raise ValueError(f"Expected {split_month_col} to exist on train_df")
 
    max_m = d[split_month_col].max()
    recent_start = (max_m - (recent_months - 1))
 
    d = d[d[split_month_col] >= recent_start].copy()
 
    y = _col_as_series(d, target_col, default=np.nan)
    y = pd.to_numeric(y, errors="coerce")
 
    # global miss rate fallback
    if global_fallback is None:
        global_miss = float((y == 0).mean()) if y.notna().any() else 0.2
    else:
        global_miss = float(global_fallback)
 
    key = _col_as_series(d, key_col, default="__MISSING__").astype(str).fillna("__MISSING__")
 
    g = pd.DataFrame({"key": key, "y": y}).dropna(subset=["y"])
    agg = g.groupby("key")["y"].agg(["count", "mean"]).reset_index()
 
    # mean is P(hit=1), we need miss_rate = 1 - mean
    hit_rate = agg["mean"].astype(float)
    miss_rate = 1.0 - hit_rate
 
    # Bayesian smoothing towards global miss
    # smoothed = (miss*count + global*alpha) / (count + alpha)
    agg["smoothed_miss_rate"] = (miss_rate * agg["count"] + global_miss * alpha) / (agg["count"] + alpha)
 
    rate_map = dict(zip(agg["key"], agg["smoothed_miss_rate"].astype(float)))
    return global_miss, rate_map
 
def apply_rate_map(
    df: pd.DataFrame,
    key_col: str,
    out_col: str,
    global_rate: float,
    rate_map: dict
) -> pd.DataFrame:
    out = df.copy()
    key = _col_as_series(out, key_col, default="__MISSING__").astype(str).fillna("__MISSING__")
    out[out_col] = key.map(rate_map).astype(float)
    out[out_col] = out[out_col].fillna(float(global_rate))
    return out
 
def add_interaction_stack_features(
    df: pd.DataFrame,
    gap_col: str = "f_lead_gap_days",
    tight_col: str = "f_is_extremely_tight",
    strict_col: str = "f_strict_tolerance",
    plant_miss_col: str = "f_plant_miss_rate",
    pressure_col: str = "f_pressure",              # if missing, we'll fallback
    plant_load_candidates=("f_plant_orders_30d","f_plant_orders_7d","f_plant_load_30d","f_plant_load_7d","f_plant_load"),
    mat_miss_col: str = "f_material_miss_rate",
    shipto_miss_col: str = "f_customer_miss_rate",   # if you have it; else it'll just use material only
) -> pd.DataFrame:
    out = df.copy()
 
    gap = pd.to_numeric(_col_as_series(out, gap_col, 0.0), errors="coerce").fillna(0.0).abs()
    tight = pd.to_numeric(_col_as_series(out, tight_col, 0), errors="coerce").fillna(0).astype(int)
    strict = pd.to_numeric(_col_as_series(out, strict_col, 0), errors="coerce").fillna(0).astype(int)
    plant_miss = pd.to_numeric(_col_as_series(out, plant_miss_col, 0.0), errors="coerce").fillna(0.0)
 
    # pressure fallback
    if pressure_col in out.columns:
        pressure = pd.to_numeric(_col_as_series(out, pressure_col, 0.0), errors="coerce").fillna(0.0)
    else:
        # fallback to something that usually exists after apply_pressure_and_bins
        pressure = pd.to_numeric(_col_as_series(out, "f_tight_ratio", 0.0), errors="coerce").fillna(0.0)
 
    # pick best available plant load column
    load = None
    for c in plant_load_candidates:
        if c in out.columns:
            load = pd.to_numeric(_col_as_series(out, c, 0.0), errors="coerce").fillna(0.0)
            break
    if load is None:
        load = pd.Series(0.0, index=out.index)
 
    mat_miss = pd.to_numeric(_col_as_series(out, mat_miss_col, 0.0), errors="coerce").fillna(0.0)
    shipto_miss = pd.to_numeric(_col_as_series(out, shipto_miss_col, 0.0), errors="coerce").fillna(0.0)
 
    # --- Requested features ---
    out["f_gap_x_load"] = gap * load
    out["f_tight_x_plant_load"] = tight * load
    out["f_strict_x_plant_miss_rate"] = strict * plant_miss
 
    # material+shipto “risk” * pressure (if shipto miss not available it contributes 0)
    out["f_mat_shipto_x_pressure"] = (mat_miss + shipto_miss) * pressure
 
    return out

In [23]:
FEATURE_COLS = list(FEATURE_COLS)  # in case it's a tuple
 
for _c in [
    "f_state_miss_rate",
    "f_gap_x_load",
    "f_tight_x_plant_load",
    "f_strict_x_plant_miss_rate",
    "f_mat_shipto_x_pressure",
]:
    if _c not in FEATURE_COLS:
        FEATURE_COLS.append(_c)

In [24]:
import shap

c:\Work\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [25]:
# =========================
# FINAL PROD PIPELINE (RECALL-FIRST MISS THRESHOLDING + Fbeta reporting)
# - Monthly retrain (rolling 12M window)
# - Train-only artifacts each window
# - Adaptive threshold tuned on previous month
# - Objective (CALIBRATION): MAX Miss Recall subject to Miss Precision >= floor
#   (fallback: best Miss F-beta if no threshold meets precision floor)
# - Reports: Miss F1 + Miss Fbeta (e.g., F2.5)
# =========================
 
import numpy as np
import pandas as pd
import lightgbm as lgb
 
from sklearn.metrics import (
    roc_auc_score, accuracy_score,
    precision_score, recall_score, f1_score,
    confusion_matrix
)
 
# -------------------------------------------------------------------
# EXPECTED TO EXIST IN YOUR NOTEBOOK / CODEBASE
# -------------------------------------------------------------------
# RANDOM_STATE
# SPLIT_DATE_COL (Requested Delivery Date based)
# TARGET_COL (0=Miss, 1=Hit)
# FEATURE_COLS (list of model feature names)
# _parse_dates, _drop_leakage, _make_target
# add_safe_features
# build_miss_rate_maps_recent, apply_miss_rate_maps
# apply_material_counts
# apply_pressure_and_bins
# build_and_apply_congestion_features
# -------------------------------------------------------------------
 
# =========================
# FINALIZED PROD SETTINGS
# =========================
PROD_LGB_PARAMS = dict(
    n_estimators=900,
    learning_rate=0.05,
    num_leaves=31,            # stable_5
    max_depth=-1,
    min_child_samples=600,    # stable_5
    subsample=0.7,            # stable_5
    colsample_bytree=0.7,     # stable_5
    reg_lambda=5.0,           # stable_5
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
 
# Recall-first threshold policy (Miss = class 0)
THRESHOLD_POLICY = dict(
    beta=2.5,                 # used for reporting + fallback scoring
    min_miss_precision=0.50,  # precision floor (lower => more recall, more false alarms)
    clamp_low=0.05,
    clamp_high=0.40,
    smooth_alpha=0.3,         # EMA smoothing
    fallback_threshold=0.34,  # first month fallback
    guardrail_max_step=0.05,  # max MoM threshold change
    min_samples=2000,         # min calibration samples
    min_miss_count=200,       # min misses in calibration set
    thresholds=np.linspace(0.01, 0.60, 120),  # search grid (finer + lower start helps recall)
    fallback_mode="best_fbeta",               # "best_fbeta" or "best_recall"
)
 
# =========================
# NEW: FEATURE HELPERS (LEAK-SAFE)
# =========================
def _safe_log1p_series(x: pd.Series) -> pd.Series:
    s = pd.to_numeric(x, errors="coerce")
    s = s.clip(lower=0)
    return np.log1p(s)
 
def fit_train_thresholds(
    train_df: pd.DataFrame,
    qty_col: str = "Ordered_Qty_in_Kgs",
    value_col: str = "f_unit_price_log",
    qty_q: float = 0.90,
    value_q: float = 0.90,
) -> dict:
    # qty threshold (p90)
    qty_s = pd.to_numeric(train_df.get(qty_col), errors="coerce")
    qty_p90 = float(qty_s.quantile(qty_q)) if qty_s.notna().any() else np.nan
 
    # value threshold (p90) - based on already engineered log price
    val_s = pd.to_numeric(train_df.get(value_col), errors="coerce")
    value_p90 = float(val_s.quantile(value_q)) if val_s.notna().any() else np.nan
 
    return {"qty_p90": qty_p90, "value_p90": value_p90}
 
def add_order_complexity_features(
    df: pd.DataFrame,
    thresholds: dict,
    qty_col: str = "Ordered_Qty_in_Kgs",
    value_col: str = "f_unit_price_log",
    extremely_tight_col: str = "f_is_extremely_tight",
) -> pd.DataFrame:
    out = df.copy()
 
    # 4.1 Quantity Magnitude
    out["f_qty_log"] = _safe_log1p_series(out.get(qty_col))
 
    # 4.2 High Quantity Flag (train p90)
    qty_thr = thresholds.get("qty_p90", np.nan)
    if pd.isna(qty_thr):
        out["f_high_qty_flag"] = 0
    else:
        out["f_high_qty_flag"] = (pd.to_numeric(out.get(qty_col), errors="coerce") >= qty_thr).astype(int)
 
    # 4.3 High Value Flag (train p90 on f_unit_price_log)
    val_thr = thresholds.get("value_p90", np.nan)
    if pd.isna(val_thr):
        out["f_high_value_flag"] = 0
    else:
        out["f_high_value_flag"] = (pd.to_numeric(out.get(value_col), errors="coerce") >= val_thr).astype(int)
 
    # 4.4 Interaction: Value × Tight
    out["f_high_value_x_tight"] = (
        out["f_high_value_flag"].astype(int)
        * pd.to_numeric(out.get(extremely_tight_col), errors="coerce").fillna(0).astype(int)
    )
 
    return out
 
def _col_as_series(df: pd.DataFrame, col: str, default=0.0) -> pd.Series:
    """
    Always return a Series aligned to df.index.
    If col missing OR df.get(col) returns a scalar, return default Series.
    """
    if col in df.columns:
        s = df[col]
        # if somehow scalar got stored, coerce to Series
        if not isinstance(s, pd.Series):
            return pd.Series(default, index=df.index)
        return s
    return pd.Series(default, index=df.index)

def add_tolerance_risk_features(
    df: pd.DataFrame,
    *,
    over_col: str,
    under_col: str,
    extremely_tight_col: str = "f_is_extremely_tight",
    gap_col: str = "f_lead_gap_days",
    strict_cutoff: float = 0.05,      # 5% strict band
    clip_band_min: float = 0.0,
    clip_band_max: float = 1.0,
) -> pd.DataFrame:
    """
    Adds tolerance-based risk features (LEAK-SAFE).
    Expects tolerance columns as either FRACTIONS (0.02=2%) or PERCENT POINTS (2=2%).
    Auto-normalizes if values look >1.
    """
    out = df.copy()
 
    over = pd.to_numeric(_col_as_series(out, over_col, default=np.nan), errors="coerce")
    under = pd.to_numeric(_col_as_series(out, under_col, default=np.nan), errors="coerce")
 
    # If stored as percent points (e.g., 2 meaning 2%), normalize to fractions
    o90 = np.nanpercentile(over.dropna(), 90) if over.notna().any() else np.nan
    u90 = np.nanpercentile(under.dropna(), 90) if under.notna().any() else np.nan
    if (not np.isnan(o90) and o90 > 1) or (not np.isnan(u90) and u90 > 1):
        over = over / 100.0
        under = under / 100.0
 
    band = (over.fillna(0.0) + under.fillna(0.0)).astype(float)
    band = band.clip(lower=clip_band_min, upper=clip_band_max)
    out["f_tolerance_band"] = band
 
    out["f_strict_tolerance"] = (out["f_tolerance_band"] < strict_cutoff).astype(np.int8)
 
    tight = pd.to_numeric(_col_as_series(out, extremely_tight_col, default=0), errors="coerce").fillna(0).astype(np.int8)
    out[extremely_tight_col] = tight
    out["f_strict_x_tight"] = (out["f_strict_tolerance"] * out[extremely_tight_col]).astype(np.int8)
 
    gap = pd.to_numeric(_col_as_series(out, gap_col, default=0.0), errors="coerce").fillna(0.0).astype(float)
    out[gap_col] = gap
    out["f_tolerance_x_gap"] = (out["f_tolerance_band"] * gap.abs()).astype(float)
 
    return out
 
 
# =========================
# OPTIONAL: UPDATE FEATURE_COLS HERE (SAFE APPEND)
# =========================
NEW_FEATURES = [
    "f_tolerance_band",
    "f_strict_tolerance",
    "f_strict_x_tight",
    "f_tolerance_x_gap",
    "f_qty_log",
    "f_high_qty_flag",
    "f_high_value_flag",
    "f_high_value_x_tight",
]
FEATURE_COLS = list(dict.fromkeys(FEATURE_COLS + NEW_FEATURES))
 
# =========================
# MONTH ITERATOR
# =========================
def month_range(start: str, end: str):
    """
    Yield pandas Periods (freq='M') from start to end inclusive.
    start/end format: 'YYYY-MM'
    """
    cur = pd.Period(start, freq="M")
    endp = pd.Period(end, freq="M")
    while cur <= endp:
        yield cur
        cur += 1
 
# =========================
# METRICS + THRESHOLD UTILS
# =========================
def fb_score(p: float, r: float, beta: float = 1.0) -> float:
    """Generic F-beta (beta>1 emphasizes recall, beta<1 emphasizes precision)."""
    if p <= 0 and r <= 0:
        return 0.0
    b2 = beta ** 2
    denom = (b2 * p + r)
    if denom == 0:
        return 0.0
    return (1 + b2) * (p * r) / denom
 
def evaluate_threshold_full(y_true, probs_hit, thr_hit: float, beta: float):
    """
    Evaluate at a given Hit threshold thr_hit.
    Also returns Miss F-beta for your chosen beta.
    """
    y_pred = (probs_hit >= thr_hit).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
 
    miss_p = precision_score(y_true, y_pred, pos_label=0, zero_division=0)
    miss_r = recall_score(y_true, y_pred, pos_label=0, zero_division=0)
    miss_f1 = f1_score(y_true, y_pred, pos_label=0, zero_division=0)
    miss_fbeta = fb_score(miss_p, miss_r, beta=beta)
 
    hit_p = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    hit_r = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
 
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, probs_hit) if len(np.unique(y_true)) > 1 else np.nan
 
    return {
        "thr_hit": float(thr_hit),
        "auc": float(auc),
        "accuracy": float(acc),
        "miss_precision": float(miss_p),
        "miss_recall": float(miss_r),
        "miss_f1": float(miss_f1),
        "miss_fbeta": float(miss_fbeta),
        "hit_precision": float(hit_p),
        "hit_recall": float(hit_r),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }
 
def find_best_threshold_max_recall_constrained(
    y_true,
    probs_hit,
    beta: float,
    min_miss_precision: float,
    thresholds,
    fallback_mode: str = "best_fbeta",  # "best_fbeta" or "best_recall"
):
    """
    RECALL-FIRST:
      Choose threshold that maximizes Miss Recall,
      subject to Miss Precision >= min_miss_precision.
    Tie-breakers (among constrained):
      1) higher miss_precision
      2) higher miss_fbeta (stability / business trade-off)
    If none satisfy precision floor:
      fallback_mode="best_fbeta": best Miss F-beta
      fallback_mode="best_recall": best Miss Recall
    """
    best_ok = None
    best_any = None
 
    for thr in thresholds:
        m = evaluate_threshold_full(y_true, probs_hit, thr, beta=beta)
        p = m["miss_precision"]
        r = m["miss_recall"]
        fb = m["miss_fbeta"]
 
        # unconstrained fallback candidate
        if best_any is None:
            best_any = {**m, "score": float(fb)}
        else:
            if fallback_mode == "best_recall":
                if r > best_any["miss_recall"]:
                    best_any = {**m, "score": float(fb)}
            else:  # best_fbeta
                if fb > best_any["score"]:
                    best_any = {**m, "score": float(fb)}
 
        # constrained: maximize recall, tie-break by precision then fbeta
        if p >= min_miss_precision:
            if best_ok is None:
                best_ok = {**m, "score": float(fb)}
            else:
                if (r > best_ok["miss_recall"]) or \
                   (r == best_ok["miss_recall"] and p > best_ok["miss_precision"]) or \
                   (r == best_ok["miss_recall"] and p == best_ok["miss_precision"] and fb > best_ok["score"]):
                    best_ok = {**m, "score": float(fb)}
 
    if best_ok is not None:
        best_ok["reason"] = "max_recall_meets_precision_floor"
        return best_ok
 
    best_any["reason"] = "no_threshold_meets_precision_floor_fallback"
    return best_any
 
def ema(prev, new, alpha=0.3):
    if prev is None:
        return float(new)
    return float(alpha * new + (1 - alpha) * prev)
 
def guardrail_threshold(prev_thr, new_thr, max_step=0.05):
    if prev_thr is None:
        return float(new_thr)
    return float(np.clip(new_thr, prev_thr - max_step, prev_thr + max_step))
 
def adaptive_threshold_from_calib(
    prev_threshold,
    calib_y_true,
    calib_probs_hit,
    policy: dict,
):
    """
    Calibrate threshold on previous month.
    Objective: MAX Miss Recall subject to Miss precision floor,
               else fallback (best_fbeta or best_recall).
    Then clamp + EMA smooth + guardrail.
    """
    beta = float(policy["beta"])
    min_samples = int(policy["min_samples"])
    min_miss_count = int(policy["min_miss_count"])
    fallback_threshold = float(policy["fallback_threshold"])
 
    thresholds = policy.get("thresholds", np.linspace(0.01, 0.60, 120))
    fallback_mode = policy.get("fallback_mode", "best_fbeta")
 
    if calib_y_true is None or len(calib_y_true) < min_samples:
        thr = prev_threshold if prev_threshold is not None else fallback_threshold
        return {"thr": float(thr), "reason": "too_few_samples"}
 
    miss_count = int((calib_y_true == 0).sum())
    if miss_count < min_miss_count:
        thr = prev_threshold if prev_threshold is not None else fallback_threshold
        return {"thr": float(thr), "reason": "too_few_misses"}
 
    best = find_best_threshold_max_recall_constrained(
        calib_y_true,
        calib_probs_hit,
        beta=beta,
        min_miss_precision=float(policy["min_miss_precision"]),
        thresholds=thresholds,
        fallback_mode=fallback_mode,
    )
 
    thr_raw = float(best["thr_hit"])
    thr_clamped = float(np.clip(thr_raw, policy["clamp_low"], policy["clamp_high"]))
    thr_smoothed = ema(prev_threshold, thr_clamped, alpha=float(policy["smooth_alpha"]))
    thr_final = guardrail_threshold(prev_threshold, thr_smoothed, max_step=float(policy["guardrail_max_step"]))
 
    return {
        "thr": float(thr_final),
        "thr_raw": float(thr_raw),
        "thr_clamped": float(thr_clamped),
        "reason": best.get("reason", "ok"),
        # calibration summary (best threshold on calib set)
        "calib_miss_precision": float(best["miss_precision"]),
        "calib_miss_recall": float(best["miss_recall"]),
        "calib_miss_f1": float(best["miss_f1"]),
        "calib_miss_fbeta": float(best["miss_fbeta"]),
        "calib_score": float(best.get("score", np.nan)),
    }

# =========================
# FINAL ROLLING EVAL
# =========================
def rolling_prod_eval(
    df_raw: pd.DataFrame,
    start_test_month: str,
    end_test_month: str,
    recent_months_missrate: int = 6,
    threshold_policy: dict = THRESHOLD_POLICY,
    verbose: bool = True,
):
    df = df_raw.copy()
    df = df_raw.copy().reset_index(drop=True)
    df["_row_id"] = np.arange(len(df))
 
    df = _parse_dates(df)
    df = _drop_leakage(df)
    df = _make_target(df)
 
    df = df.dropna(subset=[SPLIT_DATE_COL])
    df["split_month"] = df[SPLIT_DATE_COL].dt.to_period("M")
 
    month_store = {}
    train_meta = {}
 
    # 1) Train + predict per month
    for test_month in month_range(start_test_month, end_test_month):
        test_start = test_month.start_time
        test_end   = test_month.end_time
 
        train_start = (test_month - 12).start_time
        train_end   = (test_month - 1).end_time
 
        train_mask = (df[SPLIT_DATE_COL] >= train_start) & (df[SPLIT_DATE_COL] <= train_end)
        test_mask  = (df[SPLIT_DATE_COL] >= test_start) & (df[SPLIT_DATE_COL] <= test_end)
 
        train_raw = df.loc[train_mask].copy()
        test_raw  = df.loc[test_mask].copy()
 
        if len(train_raw) < 50000 or len(test_raw) == 0:
            continue
 
        # FE
        train_fe = add_safe_features(train_raw)
        test_fe  = add_safe_features(test_raw)
 
        # NEW: Train-only congestion features (leak-safe)
        train_fe, test_fe = build_and_apply_congestion_features(
            train_fe, test_fe,
            date_col="SO create date",
            windows=(7, 30),
            plant_col="Plant",
            material_col="Material",
            shipto_col="Ship_To",
        )
 
        # median fill (numerics) based on TRAIN
        numeric_cols = train_fe.select_dtypes(include=[np.number]).columns
        med = train_fe[numeric_cols].median()
        train_fe[numeric_cols] = train_fe[numeric_cols].fillna(med)
        test_fe[numeric_cols]  = test_fe[numeric_cols].fillna(med)
 
        # train-only artifacts
        global_miss, maps = build_miss_rate_maps_recent(
            train_fe, alpha=20.0, recent_months=recent_months_missrate
        )
        train_fe = apply_miss_rate_maps(train_fe, global_miss, maps)
        test_fe  = apply_miss_rate_maps(test_fe, global_miss, maps)

        # ---- NEW: Location Risk (State miss-rate) train-only artifact ----
        # NOTE: replace "State" below if your column name differs (e.g., "Ship_To_State")
        state_col = "State - Province"
        
        g_state, state_map = build_recent_smoothed_rate_map(
            train_df=train_fe,
            key_col=state_col,
            target_col=TARGET_COL,
            split_month_col="split_month",
            recent_months=recent_months_missrate,
            alpha=20.0,
            global_fallback=global_miss,   # keep consistent fallback
        )
        
        train_fe = apply_rate_map(train_fe, key_col=state_col, out_col="f_state_miss_rate", global_rate=g_state, rate_map=state_map)
        test_fe  = apply_rate_map(test_fe,  key_col=state_col, out_col="f_state_miss_rate", global_rate=g_state, rate_map=state_map)
 
        train_fe, test_fe = apply_material_counts(train_fe, test_fe)
 
        gap_q = float(train_fe["f_lead_gap_days"].quantile(0.25))
        train_fe = apply_pressure_and_bins(train_fe, gap_q)
        test_fe  = apply_pressure_and_bins(test_fe, gap_q)

        # ---- NEW: Interaction Stack (gap/load/tight/strict x plant risk, mat_shipto x pressure) ----
        train_fe = add_interaction_stack_features(train_fe)
        test_fe  = add_interaction_stack_features(test_fe)
 
        # ==========================================================
        # NEW: Tolerance-based risk + Order difficulty (LEAK-SAFE)
        # - Fit thresholds on TRAIN only
        # - Apply same thresholds to TEST
        # ==========================================================
        thresholds = fit_train_thresholds(
            train_fe,
            qty_col="Ordered_Qty_in_Kgs",
            value_col="f_unit_price_log",
            qty_q=0.90,
            value_q=0.90,
        )
 
        train_fe = add_order_complexity_features(
            train_fe,
            thresholds=thresholds,
            qty_col="Ordered_Qty_in_Kgs",
            value_col="f_unit_price_log",
            extremely_tight_col="f_is_extremely_tight",
        )
        test_fe = add_order_complexity_features(
            test_fe,
            thresholds=thresholds,
            qty_col="Ordered_Qty_in_Kgs",
            value_col="f_unit_price_log",
            extremely_tight_col="f_is_extremely_tight",
        )
 
        # NOTE: strict_cutoff assumes Overdeliv/Underdel are FRACTIONS (0.02=2%).
        # If they are in percent (2=2%), change strict_cutoff to 5.0.
        train_fe = add_tolerance_risk_features(
            train_fe,
            over_col=OVER_TOL_COL,
            under_col=UNDER_TOL_COL,
            extremely_tight_col="f_is_extremely_tight",
            gap_col="f_lead_gap_days",
            strict_cutoff=0.05,
        )
        test_fe = add_tolerance_risk_features(
            test_fe,
            over_col=OVER_TOL_COL,
            under_col=UNDER_TOL_COL,
            extremely_tight_col="f_is_extremely_tight",
            gap_col="f_lead_gap_days",
            strict_cutoff=0.05,
        )
 
        # ensure wanted features exist
        for c in FEATURE_COLS:
            if c not in train_fe.columns: train_fe[c] = np.nan
            if c not in test_fe.columns:  test_fe[c]  = np.nan
 
        X_train = train_fe[FEATURE_COLS].copy()
        y_train = train_fe[TARGET_COL].copy()
        X_test  = test_fe[FEATURE_COLS].copy()
        y_test  = test_fe[TARGET_COL].copy()
 
        feat_med = X_train.median(numeric_only=True)
        X_train = X_train.fillna(feat_med)
        X_test  = X_test.fillna(feat_med)
 
        # class weight (keep as-is)
        miss = int((y_train == 0).sum())
        hit  = int((y_train == 1).sum())
        w0 = (hit / miss) if miss > 0 else 1.0
 
        model = lgb.LGBMClassifier(**PROD_LGB_PARAMS, class_weight={0: w0, 1: 1.0})
        model.fit(X_train, y_train)
 
        probs_hit = model.predict_proba(X_test)[:, 1]

        # =========================
        # SHAP EXPLANATIONS (ROBUST VERSION)
        # =========================
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_test)

        # Handle different SHAP return formats safely
        if isinstance(shap_values, list):
            # Standard binary classifier output
            shap_miss = shap_values[0]   # class 0
            shap_hit  = shap_values[1]   # class 1
        else:
            # Some SHAP versions return only positive class explanation
            shap_hit = shap_values
            shap_miss = -shap_values     # binary symmetry


 
        mkey = str(test_month)
        month_store[mkey] = {
        "y_true": y_test.to_numpy(),
        "probs_hit": probs_hit,
        "row_ids": test_fe["_row_id"].to_numpy(),
        "X_test": X_test.copy(),
        "shap_miss": shap_miss,
        "shap_hit": shap_hit
    }

 
        train_meta[mkey] = {
            "train_end": str(train_end.date()),
            "n_train": int(len(train_raw)),
            "n_test": int(len(test_raw)),
            "global_miss_train_recent": float(global_miss),
            "gap_q_train": float(gap_q),
            "class_weight_miss_w0": float(w0),
        }
 
        if verbose:
            print(f"Stored preds for {mkey} | train_end={train_meta[mkey]['train_end']} Ntest={train_meta[mkey]['n_test']}")
 
    # 2) Adaptive threshold (calib on previous month) + evaluate
    beta = float(threshold_policy["beta"])
    months = sorted(month_store.keys())
    rows = []
    prev_thr = None
 
    for i, m in enumerate(months):
        if i == 0:
            thr_info = {"thr": float(threshold_policy["fallback_threshold"]), "reason": "first_month_fallback"}
        else:
            prev_m = months[i - 1]
            thr_info = adaptive_threshold_from_calib(
                prev_threshold=prev_thr,
                calib_y_true=month_store[prev_m]["y_true"],
                calib_probs_hit=month_store[prev_m]["probs_hit"],
                policy=threshold_policy,
            )
 
        thr = float(thr_info["thr"])
        prev_thr = thr
 
        y_true = month_store[m]["y_true"]
        probs  = month_store[m]["probs_hit"]
 
        metrics = evaluate_threshold_full(y_true, probs, thr, beta=beta)
 
        row = {
            "test_month": m,
            "thr_hit": thr,
            "thr_reason": thr_info.get("reason", ""),
            "thr_raw": thr_info.get("thr_raw", np.nan),
            "thr_clamped": thr_info.get("thr_clamped", np.nan),
            "calib_miss_precision": thr_info.get("calib_miss_precision", np.nan),
            "calib_miss_recall": thr_info.get("calib_miss_recall", np.nan),
            "calib_miss_f1": thr_info.get("calib_miss_f1", np.nan),
            "calib_miss_fbeta": thr_info.get("calib_miss_fbeta", np.nan),
            "calib_score": thr_info.get("calib_score", np.nan),
            **train_meta[m],
            **metrics,
        }
        rows.append(row)
 
        if verbose:
            print(
                f"[{m}] thr={thr:.3f} ({row['thr_reason']}) | "
                f"Miss P={row['miss_precision']:.3f} R={row['miss_recall']:.3f} "
                f"F1={row['miss_f1']:.3f} F{beta:.1f}={row['miss_fbeta']:.3f} | "
                f"AUC={row['auc']:.4f}"
                f"Hit P={row['hit_precision']:.3f} R={row['hit_recall']:.3f}"
            )


    return pd.DataFrame(rows), month_store

 
# =========================
# HOW TO RUN
# =========================
# df_monthly = rolling_prod_eval(
#     df_raw=df_raw,
#     start_test_month="2024-11",
#     end_test_month="2025-12",
#     recent_months_missrate=6,
#     threshold_policy=THRESHOLD_POLICY,
#     verbose=True,
# )
# df_monthly

In [41]:
# =========================
# RUN FINALIZED BACKTEST
# =========================
df_mar_dec, month_store = rolling_prod_eval(
    df_raw=df_raw,
    start_test_month="2024-01",
    end_test_month="2026-01",
    recent_months_missrate=6,
    threshold_policy=THRESHOLD_POLICY,
    verbose=True,
)

 


[LightGBM] [Info] Number of positive: 41106, number of negative: 13343
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.019348 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5223
[LightGBM] [Info] Number of data points in the train set: 54449, number of used features: 49
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


c:\Work\.venv\Lib\site-packages\shap\explainers\_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


Stored preds for 2024-06 | train_end=2024-05-31 Ntest=81300
[LightGBM] [Info] Number of positive: 97903, number of negative: 37846
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.016957 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5809
[LightGBM] [Info] Number of data points in the train set: 135749, number of used features: 49
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


c:\Work\.venv\Lib\site-packages\shap\explainers\_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


Stored preds for 2024-07 | train_end=2024-06-30 Ntest=77638
[LightGBM] [Info] Number of positive: 153325, number of negative: 60062
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.067739 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5939
[LightGBM] [Info] Number of data points in the train set: 213387, number of used features: 49
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


...

Stored preds for 2025-11 | train_end=2025-10-31 Ntest=51237
[LightGBM] [Info] Number of positive: 615743, number of negative: 144673
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.056294 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6328
[LightGBM] [Info] Number of data points in the train set: 760416, number of used features: 49
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


c:\Work\.venv\Lib\site-packages\shap\explainers\_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


Stored preds for 2025-12 | train_end=2025-11-30 Ntest=62733
[LightGBM] [Info] Number of positive: 626474, number of negative: 137528
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.167940 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6330
[LightGBM] [Info] Number of data points in the train set: 764002, number of used features: 49
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


c:\Work\.venv\Lib\site-packages\shap\explainers\_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


Stored preds for 2026-01 | train_end=2025-12-31 Ntest=52113
[2024-06] thr=0.340 (first_month_fallback) | Miss P=0.750 R=0.498 F1=0.599 F2.5=0.523 | AUC=0.8155Hit P=0.811 R=0.929
[2024-07] thr=0.358 (max_recall_meets_precision_floor) | Miss P=0.844 R=0.698 F1=0.764 F2.5=0.715 | AUC=0.8864Hit P=0.887 R=0.948
[2024-08] thr=0.371 (max_recall_meets_precision_floor) | Miss P=0.812 R=0.770 F1=0.791 F2.5=0.776 | AUC=0.9229Hit P=0.921 R=0.937
[2024-09] thr=0.379 (max_recall_meets_precision_floor) | Miss P=0.836 R=0.711 F1=0.768 F2.5=0.726 | AUC=0.9071Hit P=0.901 R=0.949
[2024-10] thr=0.386 (max_recall_meets_precision_floor) | Miss P=0.835 R=0.730 F1=0.779 F2.5=0.743 | AUC=0.9241Hit P=0.886 R=0.935
[2024-11] thr=0.390 (max_recall_meets_precision_floor) | Miss P=0.843 R=0.543 F1=0.661 F2.5=0.571 | AUC=0.8572Hit P=0.819 R=0.953
[2024-12] thr=0.393 (max_recall_meets_precision_floor) | Miss P=0.916 R=0.429 F1=0.584 F2.5=0.463 | AUC=0.8389Hit P=0.792 R=0.982
[2025-01] thr=0.395 (max_recall_meets_prec

In [42]:
# Per-month results (incl confusion matrix numbers)
display_cols = ["test_month","thr_hit","auc","accuracy","miss_precision","miss_recall","miss_f1","hit_precision", "hit_recall","tn","fp","fn","tp"]
df_mar_dec[display_cols]

,test_month,thr_hit,auc,accuracy,miss_precision,miss_recall,miss_f1,hit_precision,hit_recall,tn,fp,fn,tp
0,2024-06,0.340000,0.815528,0.798881,0.750492,0.498388,0.598994,0.810989,0.928517,12212,12291,4060,52737
1,2024-07,0.358000,0.886383,0.876542,0.843551,0.698010,0.763910,0.886777,0.948107,15507,6709,2876,52546
2,2024-08,0.370600,0.922930,0.894082,0.811724,0.770456,0.790552,0.920989,0.937393,15094,4497,3501,52419
3,2024-09,0.379420,0.907103,0.885931,0.835539,0.711233,0.768391,0.900684,0.949255,13448,5460,2647,49516
4,2024-10,0.385594,0.924082,0.871728,0.834550,0.730212,0.778902,0.885530,0.935137,15314,5658,3036,43770
5,2024-11,0.389916,0.857213,0.823614,0.843383,0.542810,0.660509,0.818564,0.953408,9833,8282,1826,37365
6,2024-12,0.392941,0.838871,0.809779,0.915710,0.428727,0.584020,0.791695,0.982149,7898,10524,727,39998
7,2025-01,0.395059,0.830416,0.824129,0.917666,0.475644,0.626540,0.806211,0.980813,7958,8773,714,36498
8,2025-02,0.396541,0.864723,0.855044,0.883129,0.584675,0.703559,0.848250,0.967747,8569,6087,1134,34025
9,2025-03,0.397579,0.954837,0.930917,0.893831,0.773957,0.829586,0.939510,0.974483,10330,3017,1227,46859


In [43]:
summary_mar_dec = {
    "avg_auc": float(df_mar_dec["auc"].mean()),
    "avg_accuracy": float(df_mar_dec["accuracy"].mean()),
    "avg_miss_precision": float(df_mar_dec["miss_precision"].mean()),
    "avg_miss_recall": float(df_mar_dec["miss_recall"].mean()),
    "avg_miss_f1": float(df_mar_dec["miss_f1"].mean()),
    "worst_miss_f1": float(df_mar_dec["miss_f1"].min()),
    "thr_mean": float(df_mar_dec["thr_hit"].mean()),
    "thr_std": float(df_mar_dec["thr_hit"].std(ddof=1)) if len(df_mar_dec) > 1 else 0.0,
}
summary_mar_dec

{'avg_auc': 0.9035664333707414,
 'avg_accuracy': 0.8965590737830768,
 'avg_miss_precision': 0.8382210346201798,
 'avg_miss_recall': 0.690465582573109,
 'avg_miss_f1': 0.7491640182100798,
 'worst_miss_f1': 0.5840204089178097,
 'thr_mean': 0.39000797922662966,
 'thr_std': 0.016322424762604277}

In [30]:
def export_month_predictions_with_shap(
    month_str,
    df_raw,
    month_store,
    df_metrics,
    output_path=None,
    top_n=3
):
    import numpy as np
    import pandas as pd

    if month_str not in month_store:
        raise ValueError(f"{month_str} not found in month_store")

    data = month_store[month_str]

    row_ids = data["row_ids"]
    probs_hit = data["probs_hit"]
    shap_miss = data["shap_miss"]
    shap_hit = data["shap_hit"]
    X_test = data["X_test"]

    # Get threshold used for that month
    thr = df_metrics.loc[
        df_metrics["test_month"] == month_str, "thr_hit"
    ].values[0]

    # Build base prediction DF
    df_out = df_raw.reset_index(drop=True).iloc[row_ids].copy()
    df_out["Requested Delivery Date"] = pd.to_datetime(

        df_out["Requested Delivery Date"], errors="coerce", dayfirst=True

    )
    
    print(df_out["Requested Delivery Date"].dt.to_period("M").value_counts().head(10))
 
    df_out["prob_hit"] = probs_hit
    df_out["prob_miss"] = 1 - probs_hit
    df_out["predicted_label"] = (probs_hit >= thr).astype(int)

    feature_names = X_test.columns.tolist()

    # Extract Top-N SHAP contributors (directional)
    top_features_list = []

    for i in range(len(df_out)):

        row_info = {}
        pred = df_out.iloc[i]["predicted_label"]

        # Select SHAP values based on prediction
        if pred == 0:
            row_shap = shap_miss[i]   # explain Miss
        else:
            row_shap = shap_hit[i]    # explain Hit

        # Sort descending (positive values push toward predicted class)
        sorted_idx = np.argsort(row_shap)[::-1]

        rank = 1
        count = 0

        for feat_idx in sorted_idx:
            if row_shap[feat_idx] > 0:
                row_info[f"top{rank}_feature"] = feature_names[feat_idx]
                row_info[f"top{rank}_value"] = X_test.iloc[i, feat_idx]
                row_info[f"top{rank}_shap"] = row_shap[feat_idx]

                rank += 1
                count += 1

            if count == top_n:
                break

        # If fewer than top_n positive contributors found
        while count < top_n:
            row_info[f"top{rank}_feature"] = None
            row_info[f"top{rank}_value"] = None
            row_info[f"top{rank}_shap"] = None
            rank += 1
            count += 1

        top_features_list.append(row_info)

    df_top = pd.DataFrame(top_features_list, index=df_out.index)

    df_final = pd.concat([df_out, df_top], axis=1)

    if output_path is None:
        output_path = f"OTIF_predictions_{month_str}.csv"

    df_final.to_csv(output_path, index=False)

    print(f"Saved to {output_path}")

    return df_final


In [31]:
df_dec = export_month_predictions_with_shap(
    month_str="2025-12",
    df_raw=df_raw,
    month_store=month_store,
    df_metrics=df_mar_dec
)


NameError: name 'df_mar_dec' is not defined

In [26]:
# =========================
# RUN FINALIZED BACKTEST
# =========================
df_mar_jan, month_store = rolling_prod_eval(
    df_raw=df_raw,
    start_test_month="2025-03",
    end_test_month="2026-01",
    recent_months_missrate=6,
    threshold_policy=THRESHOLD_POLICY,
    verbose=True,
)

 


[LightGBM] [Info] Number of positive: 446262, number of negative: 186815
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.169984 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6077
[LightGBM] [Info] Number of data points in the train set: 633077, number of used features: 49
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


c:\Work\.venv\Lib\site-packages\shap\explainers\_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


Stored preds for 2025-03 | train_end=2025-02-28 Ntest=61433
[LightGBM] [Info] Number of positive: 491187, number of negative: 200022
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.049212 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6121
[LightGBM] [Info] Number of data points in the train set: 691209, number of used features: 49
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


c:\Work\.venv\Lib\site-packages\shap\explainers\_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


Stored preds for 2025-04 | train_end=2025-03-31 Ntest=73507
[LightGBM] [Info] Number of positive: 549950, number of negative: 211395
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.158758 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6136
[LightGBM] [Info] Number of data points in the train set: 761345, number of used features: 49
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


...

Stored preds for 2025-11 | train_end=2025-10-31 Ntest=51237
[LightGBM] [Info] Number of positive: 615743, number of negative: 144673
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.072297 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6328
[LightGBM] [Info] Number of data points in the train set: 760416, number of used features: 49
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


c:\Work\.venv\Lib\site-packages\shap\explainers\_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


Stored preds for 2025-12 | train_end=2025-11-30 Ntest=62712
[LightGBM] [Info] Number of positive: 626461, number of negative: 137520
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.065694 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6327
[LightGBM] [Info] Number of data points in the train set: 763981, number of used features: 49
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


c:\Work\.venv\Lib\site-packages\shap\explainers\_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


Stored preds for 2026-01 | train_end=2025-12-31 Ntest=52057
[2025-03] thr=0.340 (first_month_fallback) | Miss P=0.940 R=0.753 F1=0.836 F2.5=0.774 | AUC=0.9528Hit P=0.935 R=0.987
[2025-04] thr=0.358 (max_recall_meets_precision_floor) | Miss P=0.863 R=0.825 F1=0.844 F2.5=0.830 | AUC=0.9554Hit P=0.968 R=0.976
[2025-05] thr=0.371 (max_recall_meets_precision_floor) | Miss P=0.862 R=0.744 F1=0.798 F2.5=0.758 | AUC=0.9345Hit P=0.952 R=0.977
[2025-06] thr=0.379 (max_recall_meets_precision_floor) | Miss P=0.899 R=0.759 F1=0.823 F2.5=0.775 | AUC=0.9350Hit P=0.961 R=0.986
[2025-07] thr=0.386 (max_recall_meets_precision_floor) | Miss P=0.832 R=0.761 F1=0.795 F2.5=0.770 | AUC=0.9194Hit P=0.953 R=0.969
[2025-08] thr=0.390 (max_recall_meets_precision_floor) | Miss P=0.904 R=0.669 F1=0.769 F2.5=0.694 | AUC=0.9018Hit P=0.940 R=0.987
[2025-09] thr=0.393 (max_recall_meets_precision_floor) | Miss P=0.743 R=0.741 F1=0.742 F2.5=0.742 | AUC=0.8995Hit P=0.961 R=0.961
[2025-10] thr=0.395 (max_recall_meets_prec

In [27]:
# Per-month results (incl confusion matrix numbers)
display_cols = ["test_month","thr_hit","auc","accuracy","miss_precision","miss_recall","miss_f1","hit_precision", "hit_recall","tn","fp","fn","tp"]
df_mar_jan[display_cols]

,test_month,thr_hit,auc,accuracy,miss_precision,miss_recall,miss_f1,hit_precision,hit_recall,tn,fp,fn,tp
0,2025-03,0.340000,0.952775,0.935881,0.939873,0.753053,0.836155,0.935040,0.986628,10051,3296,643,47443
1,2025-04,0.358000,0.955386,0.951937,0.863373,0.824574,0.843527,0.967572,0.975677,9523,2026,1507,60451
2,2025-05,0.370600,0.934512,0.939628,0.861563,0.743946,0.798446,0.952209,0.977106,8632,2971,1387,59196
3,2025-06,0.379420,0.935011,0.953637,0.898997,0.758754,0.822944,0.961077,0.985891,8429,2680,947,66174
4,2025-07,0.385594,0.919366,0.934776,0.831648,0.761224,0.794879,0.953256,0.969325,9376,2941,1898,59976
5,2025-08,0.389916,0.901782,0.936213,0.904023,0.668964,0.768930,0.940495,0.986608,6697,3314,711,52379
6,2025-09,0.392941,0.899460,0.932416,0.743360,0.741234,0.742295,0.960900,0.961316,5961,2081,2058,51142
7,2025-10,0.395059,0.922717,0.904399,0.629385,0.773471,0.694028,0.961636,0.925744,6764,1981,3983,49656
8,2025-11,0.396541,0.921812,0.939887,0.868394,0.732711,0.794803,0.950956,0.979024,5965,2176,904,42192
9,2025-12,0.397579,0.931775,0.921450,0.815164,0.727926,0.769079,0.941765,0.963843,8203,3066,1860,49583


In [28]:
summary_mar_jan = {
    "avg_auc": float(df_mar_jan["auc"].mean()),
    "avg_accuracy": float(df_mar_jan["accuracy"].mean()),
    "avg_miss_precision": float(df_mar_jan["miss_precision"].mean()),
    "avg_miss_recall": float(df_mar_jan["miss_recall"].mean()),
    "avg_miss_f1": float(df_mar_jan["miss_f1"].mean()),
    "worst_miss_f1": float(df_mar_jan["miss_f1"].min()),
    "thr_mean": float(df_mar_jan["thr_hit"].mean()),
    "thr_std": float(df_mar_jan["thr_hit"].std(ddof=1)) if len(df_mar_jan) > 1 else 0.0,
}
summary_mar_jan

{'avg_auc': 0.9294602142150477,
 'avg_accuracy': 0.9358948087086023,
 'avg_miss_precision': 0.8364613666283378,
 'avg_miss_recall': 0.7564369488789336,
 'avg_miss_f1': 0.7913798786501952,
 'worst_miss_f1': 0.6940283193104864,
 'thr_mean': 0.38217769577145444,
 'thr_std': 0.018873483641255198}

In [29]:
print(FEATURE_COLS)

['f_request_lead_days', 'f_material_lead_days', 'f_lead_gap_days', 'f_tight_ratio', 'f_is_tight_order', 'f_is_extremely_tight', 'f_so_to_rdd_days', 'f_mat_avail_to_rdd_days', 'f_mat_ready_after_rdd', 'f_unit_price_log', 'f_mat_total_orders_log', 'f_critical_negative_gap', 'f_mild_negative_gap', 'f_large_positive_gap', 'f_tight_x_pressure', 'f_high_plant_risk', 'f_risk_stack', 'f_otif_risk_score', 'f_gap_bin', 'f_plant_miss_rate', 'f_customer_miss_rate', 'f_material_miss_rate', 'f_gap_x_pressure', 'f_bu_miss_rate', 'f_so_woy_sin', 'f_so_woy_cos', 'f_rdd_woy_sin', 'f_rdd_woy_cos', 'f_mat_shipto_miss_rate', 'f_plant_material_miss_rate', 'f_plant_shipto_miss_rate', 'f_plant_orders_7d', 'f_plant_orders_30d', 'f_material_orders_7d', 'f_material_orders_30d', 'f_shipto_orders_7d', 'f_shipto_orders_30d', 'f_tolerance_band', 'f_strict_tolerance', 'f_strict_x_tight', 'f_tolerance_x_gap', 'f_state_miss_rate', 'f_gap_x_load', 'f_tight_x_plant_load', 'f_strict_x_plant_miss_rate', 'f_mat_shipto_x_pre

In [ ]:
['f_request_lead_days', 'f_material_lead_days', 'f_lead_gap_days', 'f_tight_ratio', 'f_is_tight_order', 'f_is_extremely_tight', 'f_so_to_rdd_days', 'f_mat_avail_to_rdd_days', 'f_mat_ready_after_rdd', 'f_unit_price_log', 'f_mat_total_orders_log', 'f_critical_negative_gap', 'f_mild_negative_gap', 'f_large_positive_gap', 'f_tight_x_pressure', 'f_high_plant_risk', 'f_risk_stack', 'f_otif_risk_score', 'f_gap_bin', 'f_plant_miss_rate', 'f_customer_miss_rate', 'f_material_miss_rate', 'f_gap_x_pressure', 'f_bu_miss_rate', 'f_so_woy_sin', 'f_so_woy_cos', 'f_rdd_woy_sin', 'f_rdd_woy_cos', 'f_mat_shipto_miss_rate', 'f_plant_material_miss_rate', 'f_plant_shipto_miss_rate', 'f_plant_orders_7d', 'f_plant_orders_30d', 'f_material_orders_7d', 'f_material_orders_30d', 'f_shipto_orders_7d', 'f_shipto_orders_30d']

In [5]:
pip install python-pptx

   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.0 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.0 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.0 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.0 MB 270.2 kB/s eta 0:00:13
   ----- ---------------------------------- 0.5/4.0 MB 270.2 kB/s eta 0:00:13
   ----- ---------------------------------- 0.5/4.0 MB 270.2 kB/s eta 0:00:13
   ----- ---------------------------------- 0.5/4.0 MB 270.2 kB/s eta 0:00:1


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import matplotlib.pyplot as plt
import shap
import pandas as pd
import numpy as np
from pptx import Presentation
from pptx.util import Inches, Pt

def generate_full_business_report(df_metrics, month_store, output_pptx="Supply_Chain_ML_Insights.pptx"):
    prs = Presentation()

    # Sort months to ensure the PPT is chronological
    sorted_months = sorted(month_store.keys())

    for m_str in sorted_months:
        # 1. DATA PREP
        data = month_store[m_str]
        X_test = data["X_test"]
        shap_miss = data["shap_miss"]  # Focused on Class 0 (Misses)
        
        # Get metrics for this specific month from your results table
        m_stats = df_metrics[df_metrics["test_month"] == m_str].iloc[0]

        # --- SLIDE 1: MONTHLY PERFORMANCE & THRESHOLD META ---
        slide = prs.slides.add_slide(prs.slide_layouts[5])
        slide.shapes.title.text = f"Executive Summary: {m_str}"
        
        # Create a Summary Table for Business Meta Data
        rows, cols = 6, 2
        table = slide.shapes.add_table(rows, cols, Inches(0.5), Inches(1.5), Inches(4), Inches(3)).table
        stats_map = [
            ("Model Strategy", "Recall-First (Miss Detection)"),
            ("Applied Threshold", f"{m_stats['thr_hit']:.3f}"),
            ("Threshold Reason", f"{m_stats['thr_reason']}"),
            ("Miss Recall (Caught)", f"{m_stats['miss_recall']:.1%}"),
            ("Miss Precision", f"{m_stats['miss_precision']:.1%}"),
            ("Test Sample Size", f"{int(m_stats['n_test'])} orders")
        ]
        for i, (k, v) in enumerate(stats_map):
            table.cell(i, 0).text = k
            table.cell(i, 1).text = v

        # --- SLIDE 2: GLOBAL FEATURE IMPORTANCE (The "Why") ---
        slide = prs.slides.add_slide(prs.slide_layouts[5])
        slide.shapes.title.text = f"Global Risk Drivers: {m_str}"
        
        # Generate SHAP Bar Plot
        plt.figure(figsize=(8, 5))
        shap.summary_plot(shap_miss, X_test, plot_type="bar", show=False)
        plt.title(f"Top Drivers of Delivery Misses - {m_str}")
        bar_path = f"tmp_bar_{m_str}.png"
        plt.savefig(bar_path, bbox_inches='tight', dpi=150)
        plt.close()
        
        slide.shapes.add_picture(bar_path, Inches(0.5), Inches(1.2), width=Inches(8.5))
        os.remove(bar_path) # Clean up

        # --- SLIDE 3: DIRECTIONAL ANALYSIS (The "How") ---
        # This shows if HIGH values increase or decrease risk
        slide = prs.slides.add_slide(prs.slide_layouts[5])
        slide.shapes.title.text = f"Impact Direction: {m_str}"

        
        
        plt.figure(figsize=(8, 5))
        shap.summary_plot(shap_miss, X_test, show=False) # Beeswarm is default
        plt.title(f"How Feature Values Push Risk - {m_str}")
        bee_path = f"tmp_bee_{m_str}.png"
        plt.savefig(bee_path, bbox_inches='tight', dpi=150)
        plt.close()

        slide.shapes.add_picture(bee_path, Inches(0.5), Inches(1.2), width=Inches(8.5))
        os.remove(bee_path)

    prs.save(output_pptx)
    print(f"Full Report Saved: {output_pptx}")

# EXECUTE
generate_full_business_report(df_mar_jan, month_store)